# TinyChirp CNN-Time TensorFlow

Train a 1D CNN on raw audio waveforms (no spectrogram), export an int8 TFLite model, and write a Rust `audio_sample.rs` file.

This mirrors `building_tensorflow/tinychirp_cnn.ipynb` but replaces the 2D mel CNN with a 1D time CNN similar to `StreamingCNNArch` from the TinyChirp `CNN_Time` model.

In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
from typing import TYPE_CHECKING
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


from building_tensorflow.utils import (  # noqa: E402
    SAMPLE_RATE,
    FRAME_LENGTH,
    FRAME_STEP,
    TARGET_FRAMES_TIME,
    TARGET_AUDIO_LEN_TIME,
    DATASET_ROOT,
    get_paths,
    configure_tf_runtime,
    set_global_seed,
)

if TYPE_CHECKING:
    import keras
else:
    from tensorflow import keras

configure_tf_runtime()
set_global_seed()

paths = get_paths("cnn_time_tf")
OUT_TFLITE = paths.out_tflite
OUT_AUDIO_RS = paths.out_audio_rs

print("Dataset root:", DATASET_ROOT)
print("Model output:", OUT_TFLITE)
print("Audio sample output:", OUT_AUDIO_RS)
print("Sample rate:", SAMPLE_RATE)
print("Frame length:", FRAME_LENGTH)
print("Frame step:", FRAME_STEP)
print("Target frames (time):", TARGET_FRAMES_TIME)
print("Target audio length (time):", TARGET_AUDIO_LEN_TIME)

Dataset root: /home/nathan/Documents/tiny-chirp-microflow/dataset
Model output: /home/nathan/Documents/tiny-chirp-microflow/models/cnn_time_tf.tflite
Audio sample output: /home/nathan/Documents/tiny-chirp-microflow/src/audio_sample.rs
Sample rate: 16000
Frame length: 1024
Frame step: 256
Target frames (time): 184
Target audio length (time): 47872


In [12]:
from building_tensorflow.utils import make_time_datasets

train_ds, val_ds, test_ds, label_names, steps_per_epoch, val_steps, test_steps = make_time_datasets()
num_labels = len(label_names)
print("Classes:", label_names)
print("Steps per epoch (train, val, test):", steps_per_epoch, val_steps, test_steps)

Found 11292 files belonging to 2 classes.
Found 1380 files belonging to 2 classes.
Found 1393 files belonging to 2 classes.
Classes: ['non_target' 'target']
Steps per epoch (train, val, test): 353 44 44


In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(TARGET_AUDIO_LEN_TIME, 1)),
    keras.layers.Conv1D(4, 3, activation="relu"),
    keras.layers.MaxPooling1D(pool_size=2, strides=2),
    keras.layers.Conv1D(8, 3, activation="relu"),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(num_labels),
])
model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=2,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps,
)
print("Test metrics:", model.evaluate(test_ds, verbose=0, steps=test_steps))

Epoch 1/2


 58/353 ━━━━━━━━━━━━━━━━━━━━ 19s 66ms/step - accuracy: 0.6759 - loss: 0.6699

In [ ]:
from building_tensorflow.utils import (
    TARGET_AUDIO_LEN_TIME,
    build_representative_batches,
    export_keras_model_to_int8_tflite,
)

# 1. Gather representative data
val_specs = build_representative_batches(test_ds, TARGET_AUDIO_LEN_TIME, take=100)

# 2–4. Export INT8 TFLite model
try:
    export_keras_model_to_int8_tflite(model, val_specs, OUT_TFLITE)
    print(f"Success! Wrote {OUT_TFLITE}")
except Exception as e:
    print(f"Conversion failed: {e}")

INFO:tensorflow:Assets written to: temp_saved_model/assets


2026-04-01 11:10:14.731430: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
INFO:tensorflow:Assets written to: temp_saved_model/assets


Saved artifact at 'temp_saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 47872, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  126071862296096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126071862294336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126071857758800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126071857759680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126071862295744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126071862295920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126071857759504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126071857760032: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1775034614.955684   35801 tf_tfl_flatbuffer_helpers.cc:390] Ignored output_format.
W0000 00:00:1775034614.955706   35801 tf_tfl_flatbuffer_helpers.cc:393] Ignored drop_control_dependency.
2026-04-01 11:10:14.955988: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: temp_saved_model
2026-04-01 11:10:14.956463: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-04-01 11:10:14.956472: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: temp_saved_model
2026-04-01 11:10:14.961958: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-04-01 11:10:14.962681: I tensorflow/cc/saved_model/loader.cc:234] Restoring SavedModel bundle.
2026-04-01 11:10:14.986757: I tensorflow/cc/saved_model/loader.cc:218] Running initialization op on SavedModel bundle at path: temp_saved_model
2026-04-01 11:10:14.993857: I tensorflow/cc/saved_model/loader.cc

Success! Wrote /home/nathan/Documents/tiny-chirp-microflow/models/cnn_time_tf.tflite


fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


In [ ]:
from building_tensorflow.utils import (
    SAMPLE_RATE,
    TARGET_AUDIO_LEN_TIME,
    collect_test_clips_for_rs,
    write_audio_sample_rs,
)

clips = collect_test_clips_for_rs(
    DATASET_ROOT / "testing",
    sample_rate=SAMPLE_RATE,
    target_len=TARGET_AUDIO_LEN_TIME,
    num_per_label=2,
)

write_audio_sample_rs(OUT_AUDIO_RS, clips, SAMPLE_RATE)
print(
    "Wrote",
    OUT_AUDIO_RS,
    "clips=",
    len(clips),
    "samples_per_clip=",
    len(clips[0][1]),
)

Found 1393 files belonging to 2 classes.


Wrote /home/nathan/Documents/tiny-chirp-microflow/src/audio_sample.rs clips= 4 samples_per_clip= 47872


/tmp/ipykernel_35801/1471352182.py:35: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  audio_vals = ", ".join(f"{float(v):.8f}" for v in audio)
